### Part 1

### 1.3. 
Investigate a parameter-efficient approach to fine-tuning this model such as Low-rank adaptation (LORA).
Please refer to the following link as to how to to use LORA with Huggingface https://huggingface.co/docs/diffusers/en/training/lora


In [21]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import BertModel

# === LoRA Config + Model Wrapper ===
class LoRABERTVoxelRegressor(nn.Module):
    def __init__(self, output_dim, r=8, alpha=32, dropout=0.1):
        super().__init__()
        base_model = BertModel.from_pretrained("bert-base-uncased")
        config = LoraConfig(
            r=r,
            lora_alpha=alpha,
            lora_dropout=dropout,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION
        )
        self.bert = get_peft_model(base_model, config)
        self.linear = nn.Linear(self.bert.config.hidden_size, output_dim)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.linear(cls)

# === Training Function (LoRA version) ===
def train_lora_bert(raw_text, fmri_data, subject_name):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    dataset = DelayedBERTDataset(raw_text, fmri_data)
    train_size = int(0.7 * len(dataset))
    val_size = int(0.15 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])
    train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32)
    test_loader = DataLoader(test_set, batch_size=32)

    output_dim = dataset[0]["target"].shape[0]
    model = LoRABERTVoxelRegressor(output_dim).to(device)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    patience = 3
    pat_counter = 0

    output_dir = f"output/lora/{subject_name}"
    os.makedirs(output_dir, exist_ok=True)
    best_model_path = os.path.join(output_dir, "bert_lora.pth")

    for epoch in range(20):
        model.train()
        train_loss = 0
        for batch in train_loader:
            inputs = tokenizer(batch["input"], padding=True, truncation=True,
                               return_tensors="pt", max_length=16)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            targets = batch["target"].to(device)
            optimizer.zero_grad()
            preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = tokenizer(batch["input"], padding=True, truncation=True,
                                   return_tensors="pt", max_length=16)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                targets = batch["target"].to(device)
                preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
                val_loss += criterion(preds, targets).item()

        print(f"[{subject_name}] Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            pat_counter = 0
        else:
            pat_counter += 1
            if pat_counter >= patience:
                print(" Early stopping")
                break

    # === Load best and evaluate ===
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs = tokenizer(batch["input"], padding=True, truncation=True,
                               return_tensors="pt", max_length=16)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            targets = batch["target"].cpu().numpy()
            preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets)

    pred = np.concatenate(all_preds, axis=0)
    true = np.concatenate(all_targets, axis=0)
    ccs = ridge_corr_pred(pred, pred, true, true, np.ones(true.shape[1]))

    np.save(os.path.join(output_dir, "voxel_cc.npy"), ccs)
    with open(os.path.join(output_dir, "summary_cc.txt"), "w") as f:
        f.write(f"Subject: {subject_name}\\n")
        f.write(f"Mean CC: {np.mean(ccs):.4f}\\n")
        f.write(f"Median CC: {np.median(ccs):.4f}\\n")
        f.write(f"Top 5%: {np.quantile(ccs, 0.95):.4f}\\n")
        f.write(f"Top 1%: {np.quantile(ccs, 0.99):.4f}\\n")

    print(f"Final mean CC: {np.mean(ccs):.4f}")
    return ccs

# === Run for both subjects ===
for name, ydata in subject_map.items():
    print(f"Running LoRA fine-tuning for {name}")
    all_ccs[name + "_lora"] = train_lora_bert(raw_text, ydata, name)

Running LoRA fine-tuning for subject2
Downsampling word sequences to TR resolution...
Creating delayed input features...
Pairing TRs with token windows...
Total usable samples: 34009
[subject2] Epoch 1 - Train Loss: 0.9893, Val Loss: 0.9834
[subject2] Epoch 2 - Train Loss: 0.9880, Val Loss: 0.9830
[subject2] Epoch 3 - Train Loss: 0.9872, Val Loss: 0.9828
[subject2] Epoch 4 - Train Loss: 0.9855, Val Loss: 0.9825
[subject2] Epoch 5 - Train Loss: 0.9828, Val Loss: 0.9837
[subject2] Epoch 6 - Train Loss: 0.9795, Val Loss: 0.9836
[subject2] Epoch 7 - Train Loss: 0.9761, Val Loss: 0.9837
 Early stopping
Final mean CC: 0.3939
Running LoRA fine-tuning for subject3
Downsampling word sequences to TR resolution...
Creating delayed input features...
Pairing TRs with token windows...
Total usable samples: 34786
[subject3] Epoch 1 - Train Loss: 0.9790, Val Loss: 0.9843
[subject3] Epoch 2 - Train Loss: 0.9778, Val Loss: 0.9840
[subject3] Epoch 3 - Train Loss: 0.9769, Val Loss: 0.9836
[subject3] Epoch

In [22]:
for name, ccs in all_ccs.items():
    print(f"== {name} Summary ==")
    print(f" Mean: {np.mean(ccs):.4f}")
    print(f" Median: {np.median(ccs):.4f}")
    print(f" Top 5% Quantile: {np.quantile(ccs, 0.95):.4f}")
    print(f" Top 1% Quantile: {np.quantile(ccs, 0.99):.4f}")

== subject2 Summary ==
 Mean: 0.4253
 Median: 0.4115
 Top 5% Quantile: 0.5123
 Top 1% Quantile: 0.5563
== subject3 Summary ==
 Mean: 0.4301
 Median: 0.4178
 Top 5% Quantile: 0.5162
 Top 1% Quantile: 0.5594
== subject2_lora Summary ==
 Mean: 0.3939
 Median: 0.3936
 Top 5% Quantile: 0.4119
 Top 1% Quantile: 0.4211
== subject3_lora Summary ==
 Mean: 0.3926
 Median: 0.3917
 Top 5% Quantile: 0.4145
 Top 1% Quantile: 0.4252
